## Autoencoders

Autoencoders are comprised of a deterministic encoder and a deterministic decoder. The encoder maps the input data to a lower-dimensional latent space, while the decoder reconstructs the original data from this latent representation. The goal of training an autoencoder is to minimize the reconstruction error between the input and the output.

The mapped vectors in the latent space are unstructured, meaning they do not have any specific distribution or organization. This can make it challenging to generate new samples from the latent space, as there is no guarantee that the latent vectors will correspond to valid data points in the original input space.

In [ ]:
import torch

In [ ]:
# Load the FashionMNIST dataset
import torchvision
import torchvision.transforms as transforms 

# Create a set of transforms to apply to the images
# a. Convert the images to PyTorch tensors
# b. Pad the 28x28 images to 32x32

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Pad(2, fill=0.0),
    transforms.Normalize((0.5,), (0.5,)),
])

trainset = torchvision.datasets.FashionMNIST(root='../data', train=True,
                                        download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=64,
                                          shuffle=True, num_workers=2)

testset = torchvision.datasets.FashionMNIST(root='../data', train=False,
                                       download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=64,
                                         shuffle=False, num_workers=2)

In [ ]:
import matplotlib.pyplot as plt

for images, labels in trainloader:
    # Plot a few images to verify the transformations
    plt.figure(figsize=(8,8))
    for i in range(4):
        plt.subplot(2, 2, i+1)
        plt.imshow(images[i].squeeze(), cmap='gray')
        plt.title(f'Label: {labels[i].item()}')
        plt.axis('off')
    plt.show()
    break

In [ ]:
## Define the encoder of the AutoEncoder
from torch import nn

class Encoder(nn.Module):
    def __init__(self):
        super(Encoder, self).__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, stride=2, padding=1)  # 32x32 -> 16x16
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1) # 16x16 -> 8x8
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1) # 8x8 -> 4x4
        self.fc = nn.Linear(64 * 4 * 4, 2)  # Flatten and reduce to latent space of size 2

    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = torch.relu(self.conv2(x))
        x = torch.relu(self.conv3(x))
        x = x.view(x.size(0), -1)  # Flatten
        x = self.fc(x)
        return x

In [ ]:
## Define the decoder of the AutoEncoder
# Use Conv2D transpose layers to upsample back to original image size

class Decoder(nn.Module):
    def __init__(self):
        super(Decoder, self).__init__()
        self.fc = nn.Linear(2, 64 * 4 * 4)  # From latent space back to feature map
        self.deconv1 = nn.ConvTranspose2d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1) # 4x4 -> 8x8
        self.deconv2 = nn.ConvTranspose2d(32, 16, kernel_size=3, stride=2, padding=1, output_padding=1) # 8x8 -> 16x16
        self.deconv3 = nn.ConvTranspose2d(16, 1, kernel_size=3, stride=2, padding=1, output_padding=1)   # 16x16 -> 32x32

    def forward(self, x):
        x = self.fc(x)  # [B, 2] -> [B, 64*4*4]
        x = x.view(x.size(0), 64, 4, 4)  # Reshape to feature map [B, 64, 4, 4] 
        x = torch.relu(self.deconv1(x))  # [B, 32, 8, 8]
        x = torch.relu(self.deconv2(x))  # [B, 16, 16, 16]
        x = torch.sigmoid(self.deconv3(x))  # Use sigmoid to get output in range [0, 1] # [B, 1, 32, 32]
        return x

In [ ]:
## Define the AutoEncoder by combining Encoder and Decoder
class AutoEncoder(nn.Module):
    def __init__(self):
        super(AutoEncoder, self).__init__()
        self.encoder = Encoder()
        self.decoder = Decoder()

    def forward(self, x):
        latent = self.encoder(x)
        reconstructed = self.decoder(latent)
        return reconstructed

In [ ]:
## Train the AutoEncoder on the FashionMNIST dataset
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f'Using device: {device}')
model = AutoEncoder().to(device)

# Loss Criterion and Optimizer
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=5e-04)

num_epochs = 10

for epoch in range(num_epochs):
    running_loss = 0.0
    for images, _ in trainloader:

        # Move images to device
        images = images.to(device)

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(images)

        # Compute loss
        loss = criterion(outputs, images)

        # Backward pass and optimization
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    epoch_loss = running_loss / len(trainloader.dataset)
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}')

In [ ]:
## Try out the trained AutoEncoder on test images
import matplotlib.pyplot as plt

for images, labels in testloader:
    # Move images to device
    images = images.to(device)
    with torch.no_grad():
        reconstructed = model(images)

        images = images.cpu()
        reconstructed = reconstructed.cpu()

        # Plot original and reconstructed images
        plt.figure(figsize=(12, 6))
        for i in range(6):
            # Original images
            plt.subplot(2, 6, i + 1)
            plt.imshow(images[i].squeeze(), cmap='gray')
            plt.title('Original')
            plt.axis('off')

            # Reconstructed images
            plt.subplot(2, 6, i + 7)
            plt.imshow(reconstructed[i].squeeze(), cmap='gray')
            plt.title('Reconstructed')
            plt.axis('off')
    break


In [ ]:
## Latent space visualization
import numpy as np
import matplotlib.pyplot as plt

latents = []
labels_list = []

for images, labels in testloader:
    images = images.to(device)
    with torch.no_grad():
        latent_vectors = model.encoder(images)
        latents.append(latent_vectors.cpu().numpy())
        labels_list.append(labels.numpy())

latents = np.concatenate(latents, axis=0)
labels_list = np.concatenate(labels_list, axis=0)

# Plot the latent space
plt.figure(figsize=(8, 6))
scatter = plt.scatter(latents[:, 0], latents[:, 1], c=labels_list, cmap='tab10', alpha=0.7, s=2)
plt.colorbar(scatter, ticks=range(10))
plt.xlabel('Latent Dimension 1')
plt.ylabel('Latent Dimension 2')
plt.title('Latent Space Visualization of FashionMNIST')
plt.grid()
plt.show()

# Get the unique set of labels
unique_labels = np.unique(labels_list)

In [ ]:
# Map from label to class names
class_names = {
    0: 'T-shirt/top',
    1: 'Trouser',
    2: 'Pullover',
    3: 'Dress',
    4: 'Coat',
    5: 'Sandal',
    6: 'Shirt',
    7: 'Sneaker',
    8: 'Bag',
    9: 'Ankle boot'
}

print("Unique class names in the latent space visualization:")
for label in unique_labels:
    print(f"{label}: {class_names[label]}")

In [ ]:
## Generate new images by sampling the latent space

# Create a grid of points in the latent space
mins, maxs = np.min(latents, axis=0), np.max(latents, axis=0)
sample = np.random.uniform(mins, maxs, size=(18, 2))

print("Sample shape:", sample.shape)

sample = torch.from_numpy(sample).float()
sample = sample.to(device)

reconstructions = model.decoder(sample)

# Plot the generated images
plt.figure(figsize=(12, 6))
for i in range(18):
    plt.subplot(3, 6, i + 1)
    plt.imshow(reconstructions[i].squeeze().detach().cpu(), cmap='gray')
    plt.title('Generated')
    plt.axis('off')
plt.show()

In [ ]:
# Generate a single image by specifying a latent vector
latent_vector = torch.tensor([[4.5, 4.0]], device=device)

reconstructed_image = model.decoder(latent_vector)
plt.figure(figsize=(4,4))
plt.imshow(reconstructed_image.squeeze().detach().cpu(), cmap='gray')
plt.title('Generated from Latent Vector [4.5, 4.0]')
plt.axis('off')
plt.show()